# 🔄 E-Commerce Big Data — ETL Pipeline (Memory-Optimized)

**Pipeline Architecture: Step 1 (Sources) → Step 2 (Ingestion) → Step 3 (Processing & Quality) → Step 4 (Storage)**

---

### Tính năng chính của ETL Pipeline:
1. **Chunk-by-Chunk Streaming Ingestion**: Đọc dữ liệu lớn hàng chục triệu bản ghi (~2.4 GB raw) theo từng chunk (100,000 rows) nhằm tối ưu hóa RAM (< 1.5 GB memory footprint).
2. **Automated Data Quality Gates**: Kiểm tra 4 chiều chất lượng (Completeness, Validity, Uniqueness, Consistency) tự động trước khi nạp vào kho lưu trữ.
3. **Data Transformation & Feature Extraction**:
   - Khử trùng lặp (Deduplication)
   - Xử lý Missing values (Imputation)
   - Chuẩn hóa chuỗi (String standardization)
   - Xác thực phiên làm việc (Session validation >= 10 ký tự)
   - Giữ giao dịch giá cao hợp lệ; loại giá không dương hoặc không hữu hạn
   - Phân tích và trích xuất cấu trúc danh mục 3 cấp (`category_level1`, `category_level2`, `category_level3`)
   - Trích xuất đặc trưng thời gian (`date`, `hour`, `day_of_week`, `day_name`, `month`, `year`)
4. **Analytical Storage & Indexing**: Nạp dữ liệu vào SQLite phân vùng chỉ mục (Indexing trên user, product, event_type, date) & lưu log metadata vào `pipeline_metadata`.
5. **Parquet Mart Export**: Xuất dữ liệu sang định dạng cột Parquet (Snappy compression) cho các tác vụ phân tích hiệu năng cao.

## 1. Cấu hình & Khởi tạo Môi trường (`src.config`, `src.storage`)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_CSV_FILES, DATA_DIR, DB_PATH, OUTPUT_DIR, RAW_DTYPES
from src.storage import init_database, get_connection, get_database_stats, export_to_parquet
from src.quality import evaluate_data_quality
from src.processing import clean_and_transform_chunk
from src.ingestion import ingest_file, ingest_all_files
from src.logger import get_logger

logger = get_logger("ETL_Notebook")
init_database()
print("✅ Database and analytical schemas successfully initialized!")
print(f"Database Location: {DB_PATH}")

## 2. Kiểm tra Nguồn dữ liệu thô (Data Sources Check)

In [ ]:
file_stats = []
for f in RAW_CSV_FILES:
    p = DATA_DIR / f
    if p.exists():
        size_mb = p.stat().st_size / (1024 * 1024)
        file_stats.append({"File Name": f, "Status": "Available", "Size (MB)": f"{size_mb:.2f} MB"})
    else:
        file_stats.append({"File Name": f, "Status": "Not Found", "Size (MB)": "0 MB"})

display(pd.DataFrame(file_stats))

## 3. Demo: Automated Data Quality Gate (`src.quality`)
Đánh giá chất lượng của một batch mẫu trước khi nạp vào hệ thống.

In [ ]:
# Đọc 50,000 dòng mẫu từ file đầu tiên để audit
sample_raw = pd.read_csv(DATA_DIR / RAW_CSV_FILES[0], nrows=50000, dtype=RAW_DTYPES, low_memory=False)
gate_passed, quality_audit = evaluate_data_quality(sample_raw, source_name="Audit_Sample_Oct")

print(f"Data Quality Gate Status: {'PASSED' if gate_passed else 'FAILED'}")
print(f"Completeness Score     : {quality_audit['completeness_score']:.2%}")
print(f"Valid Session Rate     : {quality_audit['session_valid_rate']:.2%}")
print(f"Duplicate Rate         : {quality_audit['duplicate_rate']:.2%}")
print(f"Price Valid Rate       : {quality_audit['price_valid_rate']:.2%}")

## 4. Thực thi Ingestion & Transformation (`src.ingestion`)
Chạy ingestion theo streaming chunks với chất lượng và log metadata tự động.

In [ ]:
# Chạy demo nạp dữ liệu (ở đây chạy thử nghiệm mẫu mỗi file để demo nhanh)
# Để chạy full toàn bộ dữ liệu, đặt max_chunks_per_file=None
print("Starting ETL Ingestion...")
results = ingest_all_files(max_chunks_per_file=3)

results_df = pd.DataFrame(results)
print("\n--- Ingestion Run Summary ---")
display(results_df)

## 5. Kiểm tra Trạng thái Database & Metadata Tracking

In [ ]:
stats = get_database_stats()
print(f"Total Events in Database : {stats['total_events']:,}")
print(f"Database File Size       : {stats['size_mb']:.2f} MB")
print(f"Tables                   : {stats['tables']}")

# Truy vấn 5 bản ghi mẫu đã làm sạch
conn = get_connection()
sample_clean = pd.read_sql_query("SELECT * FROM events LIMIT 5;", conn)
conn.close()

display(sample_clean)

## 6. Xuất Analytical Mart sang Định dạng Cột Parquet (`src.storage`)

In [ ]:
parquet_target = OUTPUT_DIR / "events_analytical_mart.parquet"
export_to_parquet("SELECT * FROM events LIMIT 100000;", parquet_target)
print(f"✅ Exported analytical mart to: {parquet_target}")